## F1 Score

In [1]:
import os
import re
import pandas as pd

def extract_best_metrics(log_dir, score_type):
    results = []

    # Make sure to match only validation lines
    epoch_re = re.compile(r"Epoch (\d+)")
    f1_re = re.compile(r"val.*F1:\s*([\d.]+)")
    auc_re = re.compile(r"val.*AUC-ROC:\s*([\d.]+)")
    sens_re = re.compile(r"val.*Sensitivity:\s*([\d.]+)")
    spec_re = re.compile(r"val.*Specificity:\s*([\d.]+)")
    cm_re = re.compile(r"val.*Confusion Matrix: TP=(\d+), FP=(\d+), FN=(\d+), TN=(\d+)")

    for root, _, files in os.walk(log_dir):
        print(root)
        for file in files:
            if file == "rank_0_training.log":
                path = os.path.join(root, file)
                print("Processing:", path)
                with open(path, "r") as f:
                    lines = f.readlines()

                best_score = -1
                best_metrics = None
                current_epoch = None
                temp = {}

                for line in lines:
                    # Update epoch if it's a new one
                    epoch_match = epoch_re.search(line)
                    if epoch_match:
                        current_epoch = int(epoch_match.group(1))

                    # Only consider lines that start with val
                    if "val" not in line:
                        continue

                    match = f1_re.search(line)
                    if match:
                        temp["F1 Score"] = float(match.group(1))
                        temp["Epoch"] = current_epoch
                        temp["Experiment"] = os.path.basename(root)

                    match = auc_re.search(line)
                    if match:
                        temp["ROC AUC"] = float(match.group(1))

                    match = sens_re.search(line)
                    if match:
                        temp["Sensitivity"] = float(match.group(1))

                    match = spec_re.search(line)
                    if match:
                        temp["Specificity"] = float(match.group(1))

                    match = cm_re.search(line)
                    if match:
                        temp["TP"] = int(match.group(1))
                        temp["FP"] = int(match.group(2))
                        temp["FN"] = int(match.group(3))
                        temp["TN"] = int(match.group(4))

                        if all(k in temp for k in [
                            "F1 Score", "ROC AUC", "Sensitivity", "Specificity",
                            "TP", "FP", "FN", "TN"
                        ]):
                            temp['Balanced Acc'] = (temp['Sensitivity'] + temp['Specificity'])/2
                            if temp[score_type] > best_score:
                                best_score = temp[score_type]
                                best_metrics = temp.copy()
                            temp = {}  # Reset for next epoch

                if best_metrics:
                    results.append(best_metrics)
                    print(f"Best {score_type} @", best_metrics["Epoch"], ":", best_metrics[score_type])

    return pd.DataFrame(results)

In [2]:
folder_path = "/projects/retprogression/rgarridogarcia/checkpoints"
df = extract_best_metrics(folder_path, "F1 Score")


cols = ['Experiment', 'F1 Score'] + [col for col in df.columns if col not in ['Experiment', 'F1 Score']]
df = df[cols]
df = df.sort_values(by=['F1 Score'], ascending=[False])
display(df)

df.to_csv('F1_all_experiments.csv', index=False)

# Filter conditions
oversampled_df = df[df['Experiment'].str.endswith('_oversample50')]
cropped_df     = df[df['Experiment'].str.endswith('_cropped')]
base_df        = df[~df['Experiment'].str.endswith('_oversample50') & ~df['Experiment'].str.endswith('_cropped')]

def sort_by_resolution(df):
    # Extract resolution from experiment name
    df = df.copy()
    df['resolution'] = df['Experiment'].str.extract(r'resolution(\d+)', expand=False).astype(int)
    
    # Define resolution order
    res_order = {384: 0, 512: 1, 1024: 2}
    df['res_order'] = df['resolution'].map(res_order)

    # Sort by resolution order
    df = df.sort_values(by='res_order').drop(columns=['res_order', 'resolution'])

    cols = ['Experiment', 'F1 Score'] + [col for col in df.columns if col not in ['Experiment', 'F1 Score']]
    df = df[cols]
    return df

#oversampled_df = sort_by_resolution(oversampled_df)
#cropped_df     = sort_by_resolution(cropped_df)
#base_df        = sort_by_resolution(base_df)



oversampled_df.to_csv('F1_score_oversample50.csv', index=False)
cropped_df.to_csv('F1_score_cropped.csv', index=False)
base_df.to_csv('F1_score_base.csv', index=False)

display(oversampled_df)


Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024/rank_0_training.log
Best F1 Score @ 0 : 0.9872
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50/rank_0_training.log
Best F1 Score @ 16 : 0.9753
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_/rank_0_training.log
Best F1 Score @ 16 : 0.9708
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_BrightnessFilter/rank_0_training.log
Best F1 Score @ 0 : 0.9848
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_RetinaCropped/rank_0_training.log
Best F1 Score @ 4 : 0.9684
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_RetinaCropped_BrightnessFilter/rank_0_training.log
Best F1 S

,Experiment,F1 Score,Epoch,ROC AUC,Sensitivity,Specificity,TP,FP,FN,TN,Balanced Acc
12,clean_gradable_swinv2_resolution384_oversample...,0.9897,22,0.8896,0.9915,0.5385,3958,48,34,56,0.76500
16,clean_gradable_swinv2_resolution512_oversample...,0.9888,18,0.9541,0.9877,0.6154,3943,40,49,64,0.80155
7,clean_gradable_swinv2_resolution1024_oversampl...,0.9881,2,0.9383,0.9928,0.3558,3983,67,29,37,0.67430
13,clean_gradable_swinv2_resolution384_oversample...,0.9879,13,0.9115,0.9907,0.4231,3955,60,37,44,0.70690
10,clean_gradable_swinv2_resolution384,0.9879,8,0.9459,0.9985,0.1154,3986,92,6,12,0.55695
15,clean_gradable_swinv2_resolution512_oversample50,0.9875,18,0.8888,0.9865,0.5577,3938,46,54,58,0.77210
0,clean_gradable_swinv2_resolution1024,0.9872,0,0.5420,1.0000,0.0000,4012,104,0,0,0.50000
14,clean_gradable_swinv2_resolution512,0.9871,1,0.7646,1.0000,0.0000,3992,104,0,0,0.50000
17,referable_GPU1_experiment,0.9871,0,0.9003,1.0000,0.0000,3992,104,0,0,0.50000
11,clean_gradable_swinv2_resolution384_oversample50,0.9867,15,0.8774,0.9885,0.4231,3946,60,46,44,0.70580


,Experiment,F1 Score,Epoch,ROC AUC,Sensitivity,Specificity,TP,FP,FN,TN,Balanced Acc
15,clean_gradable_swinv2_resolution512_oversample50,0.9875,18,0.8888,0.9865,0.5577,3938,46,54,58,0.77210
11,clean_gradable_swinv2_resolution384_oversample50,0.9867,15,0.8774,0.9885,0.4231,3946,60,46,44,0.70580
1,clean_gradable_swinv2_resolution1024_oversample50,0.9753,16,0.9570,0.9564,0.8173,3837,19,175,85,0.88685


## ROC AUC

In [3]:
folder_path = "/projects/retprogression/rgarridogarcia/checkpoints"
df = extract_best_metrics(folder_path, "ROC AUC")


cols = ['Experiment', 'ROC AUC'] + [col for col in df.columns if col not in ['Experiment', 'ROC AUC']]
df = df[cols]
df = df.sort_values(by=['ROC AUC'], ascending=[False])
display(df)

df.to_csv('ROC AUC_all_experiments.csv', index=False)

# Filter conditions
oversampled_df = df[df['Experiment'].str.endswith('_oversample50')]
cropped_df     = df[df['Experiment'].str.endswith('_cropped')]
base_df        = df[~df['Experiment'].str.endswith('_oversample50') & ~df['Experiment'].str.endswith('_cropped')]

def sort_by_resolution(df):
    # Extract resolution from experiment name
    df = df.copy()
    df['resolution'] = df['Experiment'].str.extract(r'resolution(\d+)', expand=False).astype(int)
    
    # Define resolution order
    res_order = {384: 0, 512: 1, 1024: 2}
    df['res_order'] = df['resolution'].map(res_order)

    # Sort by resolution order
    df = df.sort_values(by='res_order').drop(columns=['res_order', 'resolution'])

    cols = ['Experiment', 'ROC AUC'] + [col for col in df.columns if col not in ['Experiment', 'ROC AUC']]
    df = df[cols]
    return df

# oversampled_df = sort_by_resolution(oversampled_df)
# cropped_df     = sort_by_resolution(cropped_df)
# base_df        = sort_by_resolution(base_df)



oversampled_df.to_csv('ROC AUC_score_oversample50.csv', index=False)
cropped_df.to_csv('ROC AUC_score_cropped.csv', index=False)
base_df.to_csv('ROC AUC_score_base.csv', index=False)

display(oversampled_df)

Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024/rank_0_training.log
Best ROC AUC @ 8 : 0.911
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50/rank_0_training.log
Best ROC AUC @ 16 : 0.957
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_/rank_0_training.log
Best ROC AUC @ 16 : 0.9554
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_BrightnessFilter/rank_0_training.log
Best ROC AUC @ 16 : 0.9513
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_RetinaCropped/rank_0_training.log
Best ROC AUC @ 16 : 0.9743
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_RetinaCropped_BrightnessFilter/rank_0_training.log
Best ROC AUC @

,Experiment,ROC AUC,F1 Score,Epoch,Sensitivity,Specificity,TP,FP,FN,TN,Balanced Acc
4,clean_gradable_swinv2_resolution1024_oversampl...,0.9743,0.9635,16,0.9322,0.8942,3740,11,272,93,0.91320
5,clean_gradable_swinv2_resolution1024_oversampl...,0.9742,0.9550,15,0.9158,0.9231,3674,8,338,96,0.91945
6,clean_gradable_swinv2_resolution1024_oversampl...,0.9614,0.9742,16,0.9541,0.8173,3828,19,184,85,0.88570
16,clean_gradable_swinv2_resolution512_oversample...,0.9607,0.9674,11,0.9399,0.8750,3752,13,240,91,0.90745
15,clean_gradable_swinv2_resolution512_oversample50,0.9597,0.9614,6,0.9294,0.8462,3710,16,282,88,0.88780
12,clean_gradable_swinv2_resolution384_oversample...,0.9583,0.9671,3,0.9411,0.7981,3757,21,235,83,0.86960
1,clean_gradable_swinv2_resolution1024_oversample50,0.9570,0.9753,16,0.9564,0.8173,3837,19,175,85,0.88685
10,clean_gradable_swinv2_resolution384,0.9563,0.9871,8,1.0000,0.0000,3992,104,0,0,0.50000
7,clean_gradable_swinv2_resolution1024_oversampl...,0.9562,0.9859,8,0.9833,0.5577,3945,46,67,58,0.77050
8,clean_gradable_swinv2_resolution1024_oversampl...,0.9561,0.9782,16,0.9619,0.8173,3859,19,153,85,0.88960


,Experiment,ROC AUC,F1 Score,Epoch,Sensitivity,Specificity,TP,FP,FN,TN,Balanced Acc
15,clean_gradable_swinv2_resolution512_oversample50,0.9597,0.9614,6,0.9294,0.8462,3710,16,282,88,0.88780
1,clean_gradable_swinv2_resolution1024_oversample50,0.9570,0.9753,16,0.9564,0.8173,3837,19,175,85,0.88685
11,clean_gradable_swinv2_resolution384_oversample50,0.9511,0.9709,4,0.9499,0.7404,3792,27,200,77,0.84515


In [2]:
import os
import re
import pandas as pd

def extract_best_metrics(log_dir, score_type):
    results = []

    # Make sure to match only validation lines
    epoch_re = re.compile(r"Epoch (\d+)")
    f1_re = re.compile(r"val.*F1:\s*([\d.]+)")
    auc_re = re.compile(r"val.*AUC-ROC:\s*([\d.]+)")
    sens_re = re.compile(r"val.*Sensitivity:\s*([\d.]+)")
    spec_re = re.compile(r"val.*Specificity:\s*([\d.]+)")
    cm_re = re.compile(r"val.*Confusion Matrix: TP=(\d+), FP=(\d+), FN=(\d+), TN=(\d+)")

    for root, _, files in os.walk(log_dir):
        print(root)
        for file in files:
            if file == "rank_0_training.log":
                path = os.path.join(root, file)
                print("Processing:", path)
                with open(path, "r") as f:
                    lines = f.readlines()

                best_score = -1
                best_metrics = None
                current_epoch = None
                temp = {}

                for line in lines:
                    # Update epoch if it's a new one
                    epoch_match = epoch_re.search(line)
                    if epoch_match:
                        current_epoch = int(epoch_match.group(1))

                    # Only consider lines that start with val
                    if "val" not in line:
                        continue

                    match = f1_re.search(line)
                    if match:
                        temp["F1 Score"] = float(match.group(1))
                        temp["Epoch"] = current_epoch
                        temp["Experiment"] = os.path.basename(root)

                    match = auc_re.search(line)
                    if match:
                        temp["ROC AUC"] = float(match.group(1))

                    match = sens_re.search(line)
                    if match:
                        temp["Sensitivity"] = float(match.group(1))

                    match = spec_re.search(line)
                    if match:
                        temp["Specificity"] = float(match.group(1))

                    match = cm_re.search(line)
                    if match:
                        temp["TP"] = int(match.group(1))
                        temp["FP"] = int(match.group(2))
                        temp["FN"] = int(match.group(3))
                        temp["TN"] = int(match.group(4))

                        if all(k in temp for k in [
                            "F1 Score", "ROC AUC", "Sensitivity", "Specificity",
                            "TP", "FP", "FN", "TN"
                        ]):
                            temp['Balanced Acc'] = (temp['Sensitivity'] + temp['Specificity'])/2
                            if temp[score_type] > best_score:
                                best_score = temp[score_type]
                                best_metrics = temp.copy()
                            temp = {}  # Reset for next epoch

                if best_metrics:
                    results.append(best_metrics)
                    print(f"Best {score_type} @", best_metrics["Epoch"], ":", best_metrics[score_type])

    return pd.DataFrame(results)

## Balanced Accuracy

In [4]:
folder_path = "/projects/retprogression/rgarridogarcia/checkpoints"
df = extract_best_metrics(folder_path, "Balanced Acc")


cols = ['Experiment', "Balanced Acc"] + [col for col in df.columns if col not in ['Experiment', "Balanced Acc"]]
df = df[cols]
df = df.sort_values(by=["Balanced Acc"], ascending=[False])
display(df)

df.to_csv('Balanced_Acc_all_experiments.csv', index=False)

# Filter conditions
oversampled_df = df[df['Experiment'].str.endswith('_oversample50')]
cropped_df     = df[df['Experiment'].str.endswith('_cropped')]
base_df        = df[~df['Experiment'].str.endswith('_oversample50') & ~df['Experiment'].str.endswith('_cropped')]
allR_df = df[df['Experiment'].str.contains(r'_allR_\d+$')]


def sort_by_resolution(df):
    # Extract resolution from experiment name
    df = df.copy()
    df['resolution'] = df['Experiment'].str.extract(r'resolution(\d+)', expand=False).astype(int)
    
    # Define resolution order
    res_order = {384: 0, 512: 1, 1024: 2}
    df['res_order'] = df['resolution'].map(res_order)

    # Sort by resolution order
    df = df.sort_values(by='res_order').drop(columns=['res_order', 'resolution'])

    cols = ['Experiment', "Balanced Acc"] + [col for col in df.columns if col not in ['Experiment', "Balanced Acc"]]
    df = df[cols]
    return df

# oversampled_df = sort_by_resolution(oversampled_df)
# cropped_df     = sort_by_resolution(cropped_df)
# base_df        = sort_by_resolution(base_df)



'''oversampled_df.to_csv('Balanced_Acc_score_oversample50.csv', index=False)
cropped_df.to_csv('Balanced_Acc_score_cropped.csv', index=False)
base_df.to_csv('Balanced_Acc_score_base.csv', index=False)'''

display(oversampled_df)

display(allR_df)

/projects/retprogression/rgarridogarcia/checkpoints
/projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024/rank_0_training.log
Best Balanced Acc @ 0 : 0.5
/projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50/rank_0_training.log
Best Balanced Acc @ 14 : 0.88965
/projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_
Processing: /projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_/rank_0_training.log
Best Balanced Acc @ 16 : 0.88715
/projects/retprogression/rgarridogarcia/checkpoints/clean_gradable_swinv2_resolution1024_oversample50_BrightnessFilter
Processing: /projects/retprogression/rgarridogarcia

,Experiment,Balanced Acc,F1 Score,Epoch,ROC AUC,Sensitivity,Specificity,TP,FP,FN,TN
23,cropped1024_cropped_centered_wt_allR_1,0.94740,0.9812,13,0.9842,0.9646,0.9302,3843,6,141,80
19,cropped1024_cropped_centered_allR_1,0.93620,0.9814,13,0.9839,0.9654,0.9070,3846,8,138,78
21,cropped1024_cropped_centered_allR_3,0.93360,0.9829,11,0.9802,0.9691,0.8981,3830,11,122,97
5,clean_gradable_swinv2_resolution1024_oversampl...,0.93020,0.9514,8,0.9735,0.9085,0.9519,3645,5,367,99
26,cropped1024_cropped_centered_wt_allR_5,0.92615,0.9838,16,0.9853,0.9711,0.8812,3796,12,113,89
4,clean_gradable_swinv2_resolution1024_oversampl...,0.91960,0.9603,11,0.9730,0.9257,0.9135,3714,9,298,95
16,clean_gradable_swinv2_resolution512_oversample...,0.90785,0.9628,6,0.9518,0.9311,0.8846,3717,12,275,92
25,cropped1024_cropped_centered_wt_allR_4,0.90620,0.9835,17,0.9821,0.9717,0.8407,3952,18,115,95
15,clean_gradable_swinv2_resolution512_oversample50,0.90100,0.9655,23,0.9555,0.9366,0.8654,3739,14,253,90
22,cropped1024_cropped_centered_allR_4,0.89580,0.9860,13,0.9819,0.9774,0.8142,3975,21,92,92


,Experiment,Balanced Acc,F1 Score,Epoch,ROC AUC,Sensitivity,Specificity,TP,FP,FN,TN
15,clean_gradable_swinv2_resolution512_oversample50,0.90100,0.9655,23,0.9555,0.9366,0.8654,3739,14,253,90
1,clean_gradable_swinv2_resolution1024_oversample50,0.88965,0.9734,14,0.9569,0.9524,0.8269,3821,18,191,86
11,clean_gradable_swinv2_resolution384_oversample50,0.87500,0.9578,6,0.9443,0.9231,0.8269,3685,18,307,86


,Experiment,Balanced Acc,F1 Score,Epoch,ROC AUC,Sensitivity,Specificity,TP,FP,FN,TN
23,cropped1024_cropped_centered_wt_allR_1,0.94740,0.9812,13,0.9842,0.9646,0.9302,3843,6,141,80
19,cropped1024_cropped_centered_allR_1,0.93620,0.9814,13,0.9839,0.9654,0.9070,3846,8,138,78
21,cropped1024_cropped_centered_allR_3,0.93360,0.9829,11,0.9802,0.9691,0.8981,3830,11,122,97
26,cropped1024_cropped_centered_wt_allR_5,0.92615,0.9838,16,0.9853,0.9711,0.8812,3796,12,113,89
25,cropped1024_cropped_centered_wt_allR_4,0.90620,0.9835,17,0.9821,0.9717,0.8407,3952,18,115,95
22,cropped1024_cropped_centered_allR_4,0.89580,0.9860,13,0.9819,0.9774,0.8142,3975,21,92,92
24,cropped1024_cropped_centered_wt_allR_2,0.88265,0.9872,17,0.9801,0.9803,0.7850,3934,23,79,84
20,cropped1024_cropped_centered_allR_2,0.87055,0.9887,10,0.9796,0.9841,0.7570,3949,26,64,81


In [12]:
import os

def compare_folders(folder_a, folder_b):
    files_a = set(os.listdir(folder_a))
    files_b = set(os.listdir(folder_b))

    common_files = files_a & files_b
    only_in_a = files_a - files_b
    only_in_b = files_b - files_a

    print("✅ Common files:", sorted(common_files))
    print("📁 Only in A:", sorted(only_in_a))
    print("📁 Only in B:", sorted(only_in_b))

# Example usage:
folder_a = "/projects/retprogression/clean_dataset_07012025"
folder_b = "/projects/retprogression/cropped_dataset_07082025"
compare_folders(folder_a, folder_b)


✅ Common files: ['172773.jpg', '172774.jpg', '172775.jpg', '172776.jpg', '172860.jpg', '172861.jpg', '172862.jpg', '172863.jpg', '172864.jpg', '172865.jpg', '172866.jpg', '172867.jpg', '172887.jpg', '172888.jpg', '172889.jpg', '172890.jpg', '172968.jpg', '172969.jpg', '173034.jpg', '173035.jpg', '259800.jpg', '259801.jpg', '259840.jpg', '259841.jpg', '259964.jpg', '259965.jpg', '259973.jpg', '259974.jpg', '259999.jpg', '260000.jpg', '260011.jpg', '260012.jpg', '260044.jpg', '260045.jpg', '260113.jpg', '260116.jpg', '260243.jpg', '260244.jpg', '260256.jpg', '260257.jpg', '260284.jpg', '260285.jpg', '260286.jpg', '260287.jpg', '260295.jpg', '260296.jpg', '260335.jpg', '260338.jpg', '260361.jpg', '260362.jpg', '260363.jpg', '260364.jpg', '260425.jpg', '260426.jpg', '260427.jpg', '260428.jpg', '260504.jpg', '260508.jpg', '260530.jpg', '260531.jpg', '260626.jpg', '260627.jpg', '260669.jpg', '260673.jpg', '260676.jpg', '260677.jpg', '260747.jpg', '260751.jpg', '260790.jpg', '260791.jpg', '26

In [10]:
BalancedAcc_df = pd.read_csv('Balanced_Acc_all_experiments.csv')
F1_df = pd.read_csv('F1_all_experiments.csv')
ROCAUC_df = pd.read_csv('ROC AUC_all_experiments.csv')

display(BalancedAcc_df)
display(F1_df)
display(ROCAUC_df)

df1, df2, df3 = BalancedAcc_df, F1_df, ROCAUC_df


import pandas as pd

# assume you already have the three DataFrames:
# df1, df2, df3  (the ones in your screenshots)

def tag_and_align(df: pd.DataFrame) -> pd.DataFrame:
    """Add a suffix to Experiment using the column that comes right after it."""
    df = df.copy()
    # find the column immediately after 'Experiment'
    cols = list(df.columns)
    tag = cols[cols.index('Experiment') + 1]
    # append suffix to the experiment name so we can tell them apart later
    df['Experiment'] = df['Experiment'].astype(str) + f'__{tag}'
    return df

# Tag each table
df1_t = tag_and_align(df1)
df2_t = tag_and_align(df2)
df3_t = tag_and_align(df3)

# Make sure they share the same column order (use the union, then reindex each)
all_cols = list(dict.fromkeys(list(df1_t.columns) + list(df2_t.columns) + list(df3_t.columns)))
df1_t = df1_t.reindex(columns=all_cols)
df2_t = df2_t.reindex(columns=all_cols)
df3_t = df3_t.reindex(columns=all_cols)

# Merge (stack) them
merged = pd.concat([df1_t, df2_t, df3_t], ignore_index=True)

# Filter out rows with Specificity < 0.85
merged = merged[merged['Specificity'] >= 0.85]

# Sort by Sensitivity (descending)
merged = merged.sort_values(by='Sensitivity', ascending=False).reset_index(drop=True)

# Optional: view / save
display(merged)

merged.to_csv('merged_filtered_sorted.csv', index=False)


,Experiment,Balanced Acc,F1 Score,Epoch,ROC AUC,Sensitivity,Specificity,TP,FP,FN,TN
0,clean_gradable_swinv2_resolution1024_oversampl...,0.93020,0.9514,8,0.9735,0.9085,0.9519,3645,5,367,99
1,clean_gradable_swinv2_resolution1024_oversampl...,0.91960,0.9603,11,0.9730,0.9257,0.9135,3714,9,298,95
2,clean_gradable_swinv2_resolution512_oversample...,0.90785,0.9628,6,0.9518,0.9311,0.8846,3717,12,275,92
3,clean_gradable_swinv2_resolution512_oversample50,0.90100,0.9655,23,0.9555,0.9366,0.8654,3739,14,253,90
4,clean_gradable_swinv2_resolution1024_oversampl...,0.89250,0.9764,15,0.9555,0.9581,0.8269,3844,18,168,86
5,clean_gradable_swinv2_resolution1024_oversampl...,0.89185,0.9707,15,0.9607,0.9472,0.8365,3800,17,212,87
6,clean_gradable_swinv2_resolution1024_oversample50,0.88965,0.9734,14,0.9569,0.9524,0.8269,3821,18,191,86
7,clean_gradable_swinv2_resolution1024_oversampl...,0.88865,0.9573,15,0.9498,0.9215,0.8558,3697,15,315,89
8,clean_gradable_swinv2_resolution1024_oversampl...,0.88740,0.9710,16,0.9513,0.9479,0.8269,3803,18,209,86
9,clean_gradable_swinv2_resolution1024_oversampl...,0.88715,0.9708,16,0.9554,0.9474,0.8269,3801,18,211,86


,Experiment,F1 Score,Epoch,ROC AUC,Sensitivity,Specificity,TP,FP,FN,TN,Balanced Acc
0,clean_gradable_swinv2_resolution384_oversample...,0.9897,22,0.8896,0.9915,0.5385,3958,48,34,56,0.76500
1,clean_gradable_swinv2_resolution512_oversample...,0.9888,18,0.9541,0.9877,0.6154,3943,40,49,64,0.80155
2,clean_gradable_swinv2_resolution1024_oversampl...,0.9881,2,0.9383,0.9928,0.3558,3983,67,29,37,0.67430
3,clean_gradable_swinv2_resolution384_oversample...,0.9879,13,0.9115,0.9907,0.4231,3955,60,37,44,0.70690
4,clean_gradable_swinv2_resolution384,0.9879,8,0.9459,0.9985,0.1154,3986,92,6,12,0.55695
5,clean_gradable_swinv2_resolution512_oversample50,0.9875,18,0.8888,0.9865,0.5577,3938,46,54,58,0.77210
6,clean_gradable_swinv2_resolution1024,0.9872,0,0.5420,1.0000,0.0000,4012,104,0,0,0.50000
7,clean_gradable_swinv2_resolution512,0.9871,1,0.7646,1.0000,0.0000,3992,104,0,0,0.50000
8,referable_GPU1_experiment,0.9871,0,0.9003,1.0000,0.0000,3992,104,0,0,0.50000
9,clean_gradable_swinv2_resolution384_oversample50,0.9867,15,0.8774,0.9885,0.4231,3946,60,46,44,0.70580


,Experiment,ROC AUC,F1 Score,Epoch,Sensitivity,Specificity,TP,FP,FN,TN,Balanced Acc
0,clean_gradable_swinv2_resolution1024_oversampl...,0.9743,0.9635,16,0.9322,0.8942,3740,11,272,93,0.91320
1,clean_gradable_swinv2_resolution1024_oversampl...,0.9742,0.9550,15,0.9158,0.9231,3674,8,338,96,0.91945
2,clean_gradable_swinv2_resolution1024_oversampl...,0.9614,0.9742,16,0.9541,0.8173,3828,19,184,85,0.88570
3,clean_gradable_swinv2_resolution512_oversample...,0.9607,0.9674,11,0.9399,0.8750,3752,13,240,91,0.90745
4,clean_gradable_swinv2_resolution512_oversample50,0.9597,0.9614,6,0.9294,0.8462,3710,16,282,88,0.88780
5,clean_gradable_swinv2_resolution384_oversample...,0.9583,0.9671,3,0.9411,0.7981,3757,21,235,83,0.86960
6,clean_gradable_swinv2_resolution1024_oversample50,0.9570,0.9753,16,0.9564,0.8173,3837,19,175,85,0.88685
7,clean_gradable_swinv2_resolution384,0.9563,0.9871,8,1.0000,0.0000,3992,104,0,0,0.50000
8,clean_gradable_swinv2_resolution1024_oversampl...,0.9562,0.9859,8,0.9833,0.5577,3945,46,67,58,0.77050
9,clean_gradable_swinv2_resolution1024_oversampl...,0.9561,0.9782,16,0.9619,0.8173,3859,19,153,85,0.88960


,Experiment,Balanced Acc,F1 Score,Epoch,ROC AUC,Sensitivity,Specificity,TP,FP,FN,TN
0,clean_gradable_swinv2_resolution512_oversample...,0.90745,0.9674,11,0.9607,0.9399,0.8750,3752,13,240,91
1,clean_gradable_swinv2_resolution512_oversample...,0.90100,0.9655,23,0.9555,0.9366,0.8654,3739,14,253,90
2,clean_gradable_swinv2_resolution1024_oversampl...,0.91320,0.9635,16,0.9743,0.9322,0.8942,3740,11,272,93
3,clean_gradable_swinv2_resolution512_oversample...,0.90785,0.9628,6,0.9518,0.9311,0.8846,3717,12,275,92
4,clean_gradable_swinv2_resolution1024_oversampl...,0.91960,0.9603,11,0.9730,0.9257,0.9135,3714,9,298,95
5,clean_gradable_swinv2_resolution1024_oversampl...,0.88865,0.9573,15,0.9498,0.9215,0.8558,3697,15,315,89
6,clean_gradable_swinv2_resolution1024_oversampl...,0.91945,0.9550,15,0.9742,0.9158,0.9231,3674,8,338,96
7,clean_gradable_swinv2_resolution1024_oversampl...,0.93020,0.9514,8,0.9735,0.9085,0.9519,3645,5,367,99
8,clean_gradable_swinv2_resolution384_oversample...,0.88550,0.9436,2,0.9538,0.8960,0.8750,3577,13,415,91


In [20]:
import pandas as pd
import numpy as np

# Load your dataframe
BalancedAcc_df = pd.read_csv('Balanced_Acc_all_experiments.csv')

# Create a copy to store flipped metrics
flipped_df = BalancedAcc_df.copy()

# Swap the confusion matrix values
# Current: TP (minority correct), FP (majority wrong), FN (minority wrong), TN (majority correct)
# Flipped: TP becomes TN, FP becomes FN, FN becomes FP, TN becomes TP
flipped_df['TP_flipped'] = BalancedAcc_df['TN']
flipped_df['FP_flipped'] = BalancedAcc_df['FN'] 
flipped_df['FN_flipped'] = BalancedAcc_df['FP']
flipped_df['TN_flipped'] = BalancedAcc_df['TP']

# Calculate flipped metrics
precision_flipped = flipped_df['TP_flipped'] / (flipped_df['TP_flipped'] + flipped_df['FP_flipped'] + 1e-10)
precision = flipped_df['TP'] / (flipped_df['TP'] + flipped_df['FP'] + 1e-10)

# Recall (Sensitivity) for majority class
recall_flipped = flipped_df['TP_flipped'] / (flipped_df['TP_flipped'] + flipped_df['FN_flipped'] + 1e-10)
recall = flipped_df['TP'] / (flipped_df['TP'] + flipped_df['FN'] + 1e-10)

# F1 Score directly from precision and recall
flipped_df['F1_flipped'] = 2 * (precision_flipped * recall_flipped) / (precision_flipped + recall_flipped + 1e-10)
flipped_df['F1_recalculated'] = 2 * (precision * recall) / (precision + recall + 1e-10)

# Also calculate other metrics
flipped_df['Sensitivity_flipped'] = recall_flipped
flipped_df['Specificity_flipped'] = flipped_df['TN_flipped'] / (flipped_df['TN_flipped'] + flipped_df['FP_flipped'] + 1e-10)
flipped_df['Precision_flipped'] = precision_flipped

# Balanced Accuracy 
flipped_df['BalancedAcc_flipped'] = (flipped_df['Sensitivity_flipped'] + flipped_df['Specificity_flipped']) / 2

# ROC AUC remains the same
flipped_df['ROC_AUC_flipped'] = BalancedAcc_df['ROC AUC']

# Display comparison for a few key metrics
comparison_cols = ['Experiment', 'Sensitivity', 'Sensitivity_flipped', 
                   'Specificity', 'Specificity_flipped',
                   'F1 Score', 'F1_flipped']

display(flipped_df[['Experiment', 'Balanced Acc','ROC AUC',
       'Sensitivity', 'Specificity', 'Sensitivity_flipped', 'Specificity_flipped', 'F1 Score', 'F1_recalculated','F1_flipped',
       'TP', 'FP', 'FN', 'TN', 
       'TP_flipped', 'FP_flipped', 'FN_flipped', 'TN_flipped']])

flipped_df = flipped_df[['Experiment', 'Balanced Acc','ROC AUC',
       'Sensitivity', 'Specificity', 'Sensitivity_flipped', 'Specificity_flipped', 'F1 Score', 'F1_flipped',
       'TP', 'FP', 'FN', 'TN', 
       'TP_flipped', 'FP_flipped', 'FN_flipped', 'TN_flipped']]

print("Comparison of metrics (Current vs Flipped):")
print("=" * 80)
print(flipped_df[comparison_cols].to_string())

# Save to CSV if needed
flipped_df.to_csv('Balanced_Acc_with_flipped_metrics.csv', index=False)

# Quick verification for one row to check our logic
print("\n\nVerification for first row:")
print(f"Original - TP: {BalancedAcc_df.iloc[0]['TP']}, FP: {BalancedAcc_df.iloc[0]['FP']}, "
      f"FN: {BalancedAcc_df.iloc[0]['FN']}, TN: {BalancedAcc_df.iloc[0]['TN']}")
print(f"Flipped  - TP: {flipped_df.iloc[0]['TP_flipped']}, FP: {flipped_df.iloc[0]['FP_flipped']}, "
      f"FN: {flipped_df.iloc[0]['FN_flipped']}, TN: {flipped_df.iloc[0]['TN_flipped']}")
print(f"\nOriginal Sensitivity (minority): {BalancedAcc_df.iloc[0]['Sensitivity']:.4f}")
print(f"Flipped Sensitivity (majority): {flipped_df.iloc[0]['Sensitivity_flipped']:.4f}")
print(f"Original Specificity (majority): {BalancedAcc_df.iloc[0]['Specificity']:.4f}")  
print(f"Flipped Specificity (minority): {flipped_df.iloc[0]['Specificity_flipped']:.4f}")

,Experiment,Balanced Acc,ROC AUC,Sensitivity,Specificity,Sensitivity_flipped,Specificity_flipped,F1 Score,F1_recalculated,F1_flipped,TP,FP,FN,TN,TP_flipped,FP_flipped,FN_flipped,TN_flipped
0,clean_gradable_swinv2_resolution1024_oversampl...,0.93020,0.9735,0.9085,0.9519,0.951923,0.908524,0.9514,0.951449,0.347368,3645,5,367,99,99,367,5,3645
1,clean_gradable_swinv2_resolution1024_oversampl...,0.91960,0.9730,0.9257,0.9135,0.913462,0.925723,0.9603,0.960310,0.382294,3714,9,298,95,95,298,9,3714
2,clean_gradable_swinv2_resolution512_oversample...,0.90785,0.9518,0.9311,0.8846,0.884615,0.931112,0.9628,0.962829,0.390658,3717,12,275,92,92,275,12,3717
3,clean_gradable_swinv2_resolution512_oversample50,0.90100,0.9555,0.9366,0.8654,0.865385,0.936623,0.9655,0.965526,0.402685,3739,14,253,90,90,253,14,3739
4,clean_gradable_swinv2_resolution1024_oversampl...,0.89250,0.9555,0.9581,0.8269,0.826923,0.958126,0.9764,0.976378,0.480447,3844,18,168,86,86,168,18,3844
5,clean_gradable_swinv2_resolution1024_oversampl...,0.89185,0.9607,0.9472,0.8365,0.836538,0.947159,0.9707,0.970750,0.431762,3800,17,212,87,87,212,17,3800
6,clean_gradable_swinv2_resolution1024_oversample50,0.88965,0.9569,0.9524,0.8269,0.826923,0.952393,0.9734,0.973379,0.451444,3821,18,191,86,86,191,18,3821
7,clean_gradable_swinv2_resolution1024_oversampl...,0.88865,0.9498,0.9215,0.8558,0.855769,0.921486,0.9573,0.957276,0.350394,3697,15,315,89,89,315,15,3697
8,clean_gradable_swinv2_resolution1024_oversampl...,0.88740,0.9513,0.9479,0.8269,0.826923,0.947906,0.9710,0.971020,0.431078,3803,18,209,86,86,209,18,3803
9,clean_gradable_swinv2_resolution1024_oversampl...,0.88715,0.9554,0.9474,0.8269,0.826923,0.947408,0.9708,0.970757,0.428928,3801,18,211,86,86,211,18,3801


Comparison of metrics (Current vs Flipped):
                                                                          Experiment  Sensitivity  Sensitivity_flipped  Specificity  Specificity_flipped  F1 Score  F1_flipped
0   clean_gradable_swinv2_resolution1024_oversample50_RetinaCropped_BrightnessFilter       0.9085             0.951923       0.9519             0.908524    0.9514    0.347368
1                    clean_gradable_swinv2_resolution1024_oversample50_RetinaCropped       0.9257             0.913462       0.9135             0.925723    0.9603    0.382294
2                           clean_gradable_swinv2_resolution512_oversample50_cropped       0.9311             0.884615       0.8846             0.931112    0.9628    0.390658
3                                   clean_gradable_swinv2_resolution512_oversample50       0.9366             0.865385       0.8654             0.936623    0.9655    0.402685
4                      clean_gradable_swinv2_resolution1024_oversample50_moderate